In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub

In [2]:
path = kagglehub.dataset_download("sulianova/cardiovascular-disease-dataset")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\pirom\.cache\kagglehub\datasets\sulianova\cardiovascular-disease-dataset\versions\1


In [3]:
df=pd.read_csv(f'{path}\\cardio_train.csv',sep=';')
df

,id,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio
0,0,18393,2,168,62.0,110,80,1,1,0,0,1,0
1,1,20228,1,156,85.0,140,90,3,1,0,0,1,1
2,2,18857,1,165,64.0,130,70,3,1,0,0,0,1
3,3,17623,2,169,82.0,150,100,1,1,0,0,1,1
4,4,17474,1,156,56.0,100,60,1,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
69995,99993,19240,2,168,76.0,120,80,1,1,1,0,1,0
69996,99995,22601,1,158,126.0,140,90,2,2,0,0,1,1
69997,99996,19066,2,183,105.0,180,90,3,1,0,1,0,1
69998,99998,22431,1,163,72.0,135,80,1,2,0,0,0,1


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 70000 entries, 0 to 69999
Data columns (total 13 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   id           70000 non-null  int64  
 1   age          70000 non-null  int64  
 2   gender       70000 non-null  int64  
 3   height       70000 non-null  int64  
 4   weight       70000 non-null  float64
 5   ap_hi        70000 non-null  int64  
 6   ap_lo        70000 non-null  int64  
 7   cholesterol  70000 non-null  int64  
 8   gluc         70000 non-null  int64  
 9   smoke        70000 non-null  int64  
 10  alco         70000 non-null  int64  
 11  active       70000 non-null  int64  
 12  cardio       70000 non-null  int64  
dtypes: float64(1), int64(12)
memory usage: 6.9 MB


In [5]:
df.isna().sum()

id             0
age            0
gender         0
height         0
weight         0
ap_hi          0
ap_lo          0
cholesterol    0
gluc           0
smoke          0
alco           0
active         0
cardio         0
dtype: int64

In [6]:
df.duplicated().sum()

np.int64(0)

In [7]:
df['fecha_nacimiento'] = pd.to_datetime(
    df['age'], unit='D', origin='1899-12-30'
)
df['fecha_nacimiento'].head()

0   1950-05-10
1   1955-05-19
2   1951-08-17
3   1948-03-31
4   1947-11-03
Name: fecha_nacimiento, dtype: datetime64[s]

In [8]:
df['edad'] = (df['age'] / 365.25).astype(int):
df = df.drop(['age'], axis=1)

SyntaxError: invalid syntax (764871301.py, line 1)

In [ ]:
df = df[
    (df['ap_hi'] >= 80) & (df['ap_hi'] <= 220) &
    (df['ap_lo'] >= 40) & (df['ap_lo'] <= 130)
]

df = df[df['ap_hi'] > df['ap_lo']]
df.reset_index()

In [ ]:
df['imc']=df['weight']/((df['height']/100)**2)

In [ ]:
df['tipo_de_presion'] = 'bajo'

df.loc[(df['ap_hi'] > 140) & (df['ap_lo'] > 100), 'tipo_de_presion'] = 'alto'
df.loc[(df['ap_hi'] > 90) & (df['ap_lo'] > 60), 'tipo_de_presion'] = 'normal'

In [ ]:
df_cardiovascular=df[df['cardio']==1]

In [ ]:
sns.countplot(x='cardio', data=df)
plt.title("Distribución de la Variable Objetivo (Cardio)")
plt.show()
print(df['cardio'].value_counts(normalize=True))

se puede observar que el dataset esta equilibrado con casi la misma cantidad de enfermos del corazon y no

In [ ]:
sns.displot(df_cardiovascular, x="edad", hue="gender", multiple="dodge")

se puede observar que el comportamiento de las enfermedades del corazon es muy similar entre hombres y mujeres

In [ ]:
sns.displot(df_cardiovascular, x="edad", hue="tipo_de_presion", multiple="dodge")

se puede observar que el comportamiento de las enfermedades del corazon tiene una gran correlacion con la presion

In [ ]:
df_sin_fecha=df.drop(['id','fecha_nacimiento','tipo_de_presion','edad'],axis=1)

corr_matrix = df_sin_fecha.corr()

plt.figure(figsize=(10, 8))


sns.heatmap(
    corr_matrix, 
    annot=True,     
    cmap='coolwarm',
    fmt=".2f",       
    vmin=-1, vmax=1  
)

plt.title('Matriz de Correlación')
plt.show()

In [ ]:
df['tipo_de_presion'] = pd.factorize(df['tipo_de_presion'])[0]
df['tipo_de_presion'].value_counts()

In [ ]:
from sklearn.model_selection import train_test_split
X=df.drop(['cardio','fecha_nacimiento','id',],axis=1)
y=df['cardio']
# 1. Divide tus datos PRIMERO
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
num_cols = ['height', 'weight', 'ap_hi', 'ap_lo', 'imc']

preprocessor = ColumnTransformer(
    transformers=[
        ('scaler', StandardScaler(), num_cols) 
    ],
    remainder='passthrough'
)


X_train_scaled_array = preprocessor.fit_transform(X_train)


X_test_scaled_array = preprocessor.transform(X_test)


columnas_ordenadas = num_cols + [col for col in X.columns if col not in num_cols]

X_train = pd.DataFrame(X_train_scaled_array, columns=columnas_ordenadas, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled_array, columns=columnas_ordenadas, index=X_test.index)

# ¡Listo! Ya puedes usar X_train_scaled para entrenar tu modelo
print("¡Datos escalados con éxito!")

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Initialize the classifier with 100 trees
clf = RandomForestClassifier()

# Train the model
clf.fit(X_train, y_train)


In [ ]:
y_pred = clf.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
matriz = confusion_matrix(y_test, y_pred)

print('accuracy = ',accuracy)
print('precision = ',precision)
print('recall = ', recall)
print('f1 = ',f1)
print('matriz = ',matriz)

In [ ]:
import xgboost as xgb
model_xgb = xgb.XGBClassifier()
model_xgb.fit(X_train, y_train)


In [ ]:
y_pred_xgb = model_xgb.predict(X_test)

In [ ]:
accuracy = accuracy_score(y_test, y_pred_xgb)
precision = precision_score(y_test, y_pred_xgb)
recall = recall_score(y_test, y_pred_xgb)
f1 = f1_score(y_test, y_pred_xgb)
matriz = confusion_matrix(y_test, y_pred_xgb)

print('accuracy = ',accuracy)
print('precision = ',precision)
print('recall = ', recall)
print('f1 = ',f1)
print('matriz = ',matriz)

In [ ]:
import lightgbm as lgb
model_lgb = lgb.LGBMClassifier()
model_lgb.fit(X_train, y_train)

In [ ]:
y_pred_lgb = model_lgb.predict(X_test)

In [ ]:
accuracy = accuracy_score(y_test, y_pred_lgb)
precision = precision_score(y_test, y_pred_lgb)
recall = recall_score(y_test, y_pred_lgb)
f1 = f1_score(y_test, y_pred_lgb)
matriz = confusion_matrix(y_test, y_pred_lgb)

print('accuracy = ',accuracy)
print('precision = ',precision)
print('recall = ', recall)
print('f1 = ',f1)
print('matriz = ',matriz)

In [ ]:
param_dist = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [3, 5, 7, 10],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'gamma': [0, 0.1, 0.2],
    'min_child_weight': [1, 3, 5]
}

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
random_search = RandomizedSearchCV(
    estimator=model_xgb,
    param_distributions=param_dist,
    n_iter=20,  
    scoring='recall',
    cv=5,
    verbose=2,
    n_jobs=-1,
    random_state=42
)

In [ ]:
random_search.fit(X_train, y_train)

In [ ]:

from sklearn.metrics import roc_auc_score
best_model = random_search.best_estimator_

y_proba = best_model.predict_proba(X_test)[:, 1]

y_pred = (y_proba > 0.3).astype(int)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
matriz = confusion_matrix(y_test, y_pred)
roc_auc=roc_auc_score(y_test, y_proba)

print('accuracy = ',accuracy)
print('precision = ',precision)
print('recall = ', recall)
print('f1 = ',f1)
print('roc_auc = ',roc_auc)
print('matriz = ',matriz)

In [ ]:
from sklearn.metrics import RocCurveDisplay
RocCurveDisplay.from_predictions(y_test, y_proba)

In [ ]:
xgb.plot_importance(best_model, importance_type='gain')
plt.show()

In [ ]:
import shap

explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test)

shap.summary_plot(shap_values, X_test)